# Cost optimisation algorithms

**Data sources:** 

* production - solarEdge (ID:1567917)
* consumption - mvm (GroupID: 4)
* price - HUPX.hu



Using cassandra to store the aforementioned sources after transformations: 
* **consumer_profile**
*  **solar_panel_production**
*  **hupx_energy_price**

Data is stored in 15 minute time intervals. production and consumption is stored in **kwh**, price is in **mwh**



In [37]:
spark.version

'3.5.0'

# Setup

In [1]:
import findspark
import itertools
import pyspark.sql
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (row_number, abs, weekofyear, to_timestamp, 
                                   col, to_utc_timestamp, count, max, 
                                   monotonically_increasing_id, lit, concat, expr, 
                                   regexp_replace,lpad)
 

findspark.init()

spark = SparkSession.builder \
    .appName("CassandraConncection") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0")\
    .config("spark.sql.catalog.client","com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.client.spark.cassandra.connection.host","127.0.0.1")\
    .getOrCreate()
    #.config("spark.cassandra.connection.port", "9042")\
    


24/07/02 12:32:00 WARN Utils: Your hostname, PySpark resolves to a loopback address: 127.0.1.1; using 192.168.64.129 instead (on interface ens33)
24/07/02 12:32:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/student/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/student/.ivy2/cache
The jars for the packages stored in: /home/student/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-87bb164a-9cfa-4e8c-88e0-a452e534f451;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.0 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.0 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found com.datastax.oss#java-driver-core-shaded;4.13.0 in central
	found com.datastax.oss#native-protocol;1.5.0 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3

# Extract
Extracting data from csv files

In [119]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show()
price.show()






+----------+----------+--------+
|     Dátum|Negyedórák| Group#4|
+----------+----------+--------+
|2022.01.01|      0:00|0.018103|
|2022.01.01|      0:15|0.018038|
|2022.01.01|      0:30|0.018010|
|2022.01.01|      0:45|0.018095|
|2022.01.01|      1:00|0.018187|
|2022.01.01|      1:15|0.018158|
|2022.01.01|      1:30|0.018087|
|2022.01.01|      1:45|0.018103|
|2022.01.01|      2:00|0.018172|
|2022.01.01|      2:15|0.018167|
|2022.01.01|      2:30|0.018129|
|2022.01.01|      2:45|0.018056|
|2022.01.01|      3:00|0.017923|
|2022.01.01|      3:15|0.017764|
|2022.01.01|      3:30|0.017725|
|2022.01.01|      3:45|0.017770|
|2022.01.01|      4:00|0.017912|
|2022.01.01|      4:15|0.018011|
|2022.01.01|      4:30|0.018170|
|2022.01.01|      4:45|0.018460|
+----------+----------+--------+
only showing top 20 rows

+----------------+--------------+
|            time|production_(W)|
+----------------+--------------+
|01.01.2022 00:00|             0|
|01.01.2022 00:15|             0|
|01.01.2022 

# Transform

Transformations are needed: All timezones should be casted to **UTC**, granuality should be set to 15 minutes to all sources

production profile:
+ solar panel ID from external source is added manually
+ production in Watt is scaled to kwh

In [120]:
spark.conf.set("spark.sql.session.timeZone", "UTC") # necessary config to aviod clock shift
production_final = production.withColumn("timestamp",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                        .withColumn("profile_id", lit(1567917))\
                        .withColumn("production_(W)", col("production_(W)")*0.00025)\
                        .withColumnRenamed("production_(W)", "production_kwh")\
                        .select("profile_id","timestamp","production_kwh")
                        


consumption profile:

+ consumption is scaled to a yearly 10k kwh demand
+ profile id is set to 4 as the business group 4#'s consumption profile was selected from the mvm chart.

In [121]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
cons_final = consumption.withColumn("time", to_timestamp(concat(col("Dátum"), lit(" "), col("Negyedórák")), "yyyy.MM.dd HH:mm"))\
                        .withColumn("utc_time", to_utc_timestamp(col("time"), "Europe/Budapest"))\
                        .withColumn("consumption_kwh", col("Group#4").cast("double")*10)\
                        .withColumn("rownum", monotonically_increasing_id())

cons_filter = cons_final.groupBy("utc_time").agg(count("*").alias("count")).filter(col("count")>=2)
cons_filtered = cons_final.join(cons_filter,"utc_time","inner").groupBy("utc_time").agg(max("rownum").alias("rownum"))\
                          .withColumn("modified_utc_time", expr("utc_time + INTERVAL 1 HOUR"))\
                          .select("modified_utc_time","rownum")

cons_final = cons_final.join(cons_filtered,"rownum","left_outer")\
                       .withColumn("timestamp", expr("IFNULL(modified_utc_time, utc_time)"))\
                       .withColumn("profile_id", lit(4))\
                       .select("profile_id","timestamp","consumption_kwh")

cons_final.show()

+----------+-------------------+-------------------+
|profile_id|          timestamp|    consumption_kwh|
+----------+-------------------+-------------------+
|         4|2021-12-31 23:00:00|0.18103000000000002|
|         4|2021-12-31 23:15:00|0.18037999999999998|
|         4|2021-12-31 23:30:00|             0.1801|
|         4|2021-12-31 23:45:00|            0.18095|
|         4|2022-01-01 00:00:00|0.18186999999999998|
|         4|2022-01-01 00:15:00|0.18158000000000002|
|         4|2022-01-01 00:30:00|0.18086999999999998|
|         4|2022-01-01 00:45:00|0.18103000000000002|
|         4|2022-01-01 01:00:00|            0.18172|
|         4|2022-01-01 01:15:00|            0.18167|
|         4|2022-01-01 01:30:00|            0.18129|
|         4|2022-01-01 01:45:00|            0.18056|
|         4|2022-01-01 02:00:00|            0.17923|
|         4|2022-01-01 02:15:00|            0.17764|
|         4|2022-01-01 02:30:00|0.17725000000000002|
|         4|2022-01-01 02:45:00|0.177700000000

price:

+ each hourly price entry is multiplied by 4 to match prices to each quarter hours.

In [122]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
price_final = price.withColumn("Hours",regexp_replace("Hours","[HB]",""))\
                   .withColumn("Hours", expr("cast(Hours as int) - 1"))\
                   .withColumn("Quarters",expr("explode(array(':00',':15',':30',':45'))"))\
                   .withColumn("Hours", concat(lpad("Hours",2,"0"),"Quarters"))\
                   .withColumn("time",to_timestamp(concat("Delivery day",lit(" "),"Hours"),"dd.MM.yyyy HH:mm"))\
                   .withColumn("utc_timestamp", to_utc_timestamp("time", "Europe/Budapest"))\
                   .withColumn("rownum", monotonically_increasing_id())

price_filter = price_final.groupBy("utc_timestamp").agg(count("*").alias("count")).filter(col("count")>=2)
price_filtered = price_final.join(price_filter,"utc_timestamp","inner").groupBy("utc_timestamp").agg(max("rownum").alias("rownum"))\
                            .withColumn("modified_utc_timestamp", expr("utc_timestamp + INTERVAL 1 HOUR")).select("modified_utc_timestamp","rownum")

price_final = price_final.join(price_filtered,"rownum","left_outer")\
                         .withColumn("timestamp", expr("IFNULL(modified_utc_timestamp, utc_timestamp)"))\
                         .withColumnRenamed("Prices (EUR/Mwh)","price_eur_per_mwh")\
                         .select("timestamp","price_eur_per_mwh")

                   

price_final.show()
price_final.printSchema()
price_final.count()

+-------------------+-----------------+
|          timestamp|price_eur_per_mwh|
+-------------------+-----------------+
|2021-12-31 23:00:00|            61.84|
|2021-12-31 23:15:00|            61.84|
|2021-12-31 23:30:00|            61.84|
|2021-12-31 23:45:00|            61.84|
|2022-01-01 00:00:00|            41.33|
|2022-01-01 00:15:00|            41.33|
|2022-01-01 00:30:00|            41.33|
|2022-01-01 00:45:00|            41.33|
|2022-01-01 01:00:00|            43.22|
|2022-01-01 01:15:00|            43.22|
|2022-01-01 01:30:00|            43.22|
|2022-01-01 01:45:00|            43.22|
|2022-01-01 02:00:00|            45.46|
|2022-01-01 02:15:00|            45.46|
|2022-01-01 02:30:00|            45.46|
|2022-01-01 02:45:00|            45.46|
|2022-01-01 03:00:00|            37.67|
|2022-01-01 03:15:00|            37.67|
|2022-01-01 03:30:00|            37.67|
|2022-01-01 03:45:00|            37.67|
+-------------------+-----------------+
only showing top 20 rows

root
 |-- time

70080

# Load
Loading transformed data into cassandra

In [6]:
cons_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="consumption_profile", keyspace="energycom") \
    .save()

In [7]:
production_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="production_profile", keyspace="energycom") \
    .save()

In [8]:
price_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .save()

# Using csv data (optional)

In [123]:
consumption = cons_final
production = production_final
price = price_final

# Reading from Cassandra DB


In [220]:
consumption = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumption_profile", keyspace="energycom") \
    .load()
production = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="production_profile", keyspace="energycom") \
    .load()
price = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .load()

consumption.show()
production.show()
price.show()

+-------------------+----------+-------------------+
|          timestamp|profile_id|    consumption_kwh|
+-------------------+----------+-------------------+
|2023-11-13 14:45:00|         4|            0.60593|
|2022-03-01 00:30:00|         4|0.15866000000000002|
|2023-11-13 04:30:00|         4|0.22551000000000002|
|2022-06-08 01:15:00|         4|0.12174000000000001|
|2022-02-12 18:30:00|         4|0.21694999999999998|
|2022-04-27 08:00:00|         4| 0.6915399999999999|
|2022-11-04 08:00:00|         4|            0.85802|
|2022-11-26 09:45:00|         4|            0.17558|
|2022-09-08 04:30:00|         4|            0.40088|
|2023-08-21 12:30:00|         4|            0.21502|
|2022-12-19 06:30:00|         4|            0.69099|
|2023-09-02 21:30:00|         4|0.12491000000000001|
|2023-05-26 14:00:00|         4|            0.36662|
|2023-01-31 02:45:00|         4|            0.18899|
|2023-06-03 13:30:00|         4|            0.12864|
|2023-11-06 18:15:00|         4|            0.

# Scaling production

In [221]:
#production = production.withColumn("production_kwh", col("production_kwh")*50/10.8) # ~50 kwh yearly production
production = production.withColumn("production_kwh", col("production_kwh")*8/10.8)  # ~ 8 kwh yearly production
#production = production.withColumn("production_kwh", col("production_kwh")*6/10.8)  # ~ 6 kwh yearly production
#production = production.withColumn("production_kwh", col("production_kwh")*4/10.8)   # ~ 4 kwh yearly production

#num = production.toPandas()["production_kwh"].sum()/50/2
#num = production.toPandas()["production_kwh"].sum()/10/2
#num = production.toPandas()["production_kwh"].sum()/8/2
#num = production.toPandas()["production_kwh"].sum()/4/2

all = production.toPandas()["production_kwh"].sum()/2
#print(f" normalized production {num} kwh per year")
print(f" nominal production {all} kwh per year")


 nominal production 10522.540243990741 kwh per year


# Combining data
the dataframes created from the sources are united into a single dataframe for further use

In [222]:
from pyspark.sql.functions import *

# filter
consumption   = consumption.filter(col("profile_id") == 4)
production = production.filter(col("profile_id") == 1567917)

# join
process = consumption.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#sort and cast to Pandas
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()

process.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2022-01-01 00:00:00,0.18187,0.0,41.330002
1,2022-01-01 00:15:00,0.18158,0.0,41.330002
2,2022-01-01 00:30:00,0.18087,0.0,41.330002
3,2022-01-01 00:45:00,0.18103,0.0,41.330002
4,2022-01-01 01:00:00,0.18172,0.0,43.220001


**Optional filtering**

In [223]:
process = process[ process["timestamp"]< "2023.01.01"]
process.tail()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
35035,2022-12-31 22:45:00,0.15594,0.0,2.06
35036,2022-12-31 23:00:00,0.16908,0.0,19.76
35037,2022-12-31 23:15:00,0.18221,0.0,19.76
35038,2022-12-31 23:30:00,0.18155,0.0,19.76
35039,2022-12-31 23:45:00,0.18127,0.0,19.76


# Optimistaion strategies

Goal: minimise **energy costs**

Main measurements: 
+ **energy cost**
+ **self-consumption rate**
+ **self-sufficiency rate**
+ net renvenue from sales
  

Scenarios:
* pv only
* greedy strategy
* rule based strategy full
* rule based strategy without LT
* linear programming optimising net revenue
* linear programming optimising only energy cost

# System parameters
Parameters are set in this block for the test environment including:
* SOC_MIN     - (minimum state of charge allowed for BESS)
* SOC_MAX     - (maximum charge capacity for BESS)
* CHARGE_RATE - (maximum possible charge rate during time period of 15 minutes)
* CHARGE_RATE - (maximum possible discharge rate during time period of 15 minutes)
* START       - (BESS's starting charge state)

In [9]:
SOC_MIN = 0
SOC_MAX = 50
CHARGE_RATE= 12.5
DISCHARGE_RATE = 12.5
START = 0

# PV only

There is no BESS in this scenario, if there is a sufficit, we sell it to the grid, if there is a power deficit, we buy from the grid.

In [211]:
pv = process.assign(c_out = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                    c_in   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                    sell_price = lambda x: x["c_out"]*x["price_eur_per_mwh"]*0.00025,
                    buy_price = lambda x: x["c_in"]*x["price_eur_per_mwh"]*0.00025,
                    net_revenue = lambda x: x["sell_price"] - x["buy_price"])

pv.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,c_out,c_in,sell_price,buy_price,net_revenue
0,2022-01-01 00:00:00,0.18187,0.0,41.330002,0.0,0.18187,0.0,0.001879,-0.001879
1,2022-01-01 00:15:00,0.18158,0.0,41.330002,0.0,0.18158,0.0,0.001876,-0.001876
2,2022-01-01 00:30:00,0.18087,0.0,41.330002,0.0,0.18087,0.0,0.001869,-0.001869
3,2022-01-01 00:45:00,0.18103,0.0,41.330002,0.0,0.18103,0.0,0.001870,-0.001870
4,2022-01-01 01:00:00,0.18172,0.0,43.220001,0.0,0.18172,0.0,0.001963,-0.001963


In [212]:
pv_total_generation = pv["production_kwh"].sum()
pv_total_export = pv["c_out"].sum()
pv_self_consumption_rate = 1 - pv_total_export/pv_total_generation

pv_total_consumption = pv["consumption_kwh"].sum()
pv_total_import = pv["c_in"].sum()
pv_self_sufficiency_rate = 1 - pv_total_import/pv_total_consumption

total_cost = pv['buy_price'].sum()
total_revenue = pv['sell_price'].sum()
net_revenue = pv['net_revenue'].sum()

print(f"Total generation: {pv_total_generation}")
print(f"Total consumption: {pv_total_consumption}")
print(f"Self consumption rate: {pv_self_consumption_rate}")
print(f"Self sufficiency_rate: {pv_self_sufficiency_rate}")
print(f"Total cost: {total_cost}")
print(f"Total revenue: {total_revenue}")
print(f"net revenue: {net_revenue}")


Total generation: 38.948577880859375
Total consumption: 172.95486450195312
Self consumption rate: 0.566851407289505
Self sufficiency_rate: 0.127652108669281
Total cost: 5.146556377410889
Total revenue: 0.3355187177658081
net revenue: -4.811038017272949


# Greedy strategy

If production covers consumption, then surplus is stored into BESS, if it is full, then the remainder is sold to the grid.
If production does not cover consumption, then the deficit is first covered from BESS, and if the energy stored in BESS was not sufficient, then power is bought from the grid.



In [213]:
import numpy as np
# calculating energy surplus and deficit from the difference of production and consumption,
# SOC colums are added for the beggining and end of 15 minute intervals, setting default value to 0. 
basic_p = process.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                       deficit   = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                       SOC_start = float(0.00000),
                       SOC_end   = float(0.00000))
#basic_p.head()

# setting the starting state of the battery:
basic_p.at[0,"SOC_start"] = START

#calculating the amount we can store of the surplus generated
if basic_p.at[0,"surplus"] > 0:
        basic_p.at[0,"SOC_end"] = __builtins__.min(basic_p.at[0,"SOC_start"] + basic_p.at[0,"surplus"]\
                                                   ,basic_p.at[0,"SOC_start"] + CHARGE_RATE\
                                                   ,SOC_MAX)
       
            
#calculating how much of the deficit we can cover from our battery
else:
        basic_p.at[0,"SOC_end"] = __builtins__.max(basic_p.at[0,"SOC_start"] - basic_p.at[0,"deficit"]\
                                                   ,basic_p.at[0,"SOC_start"] - DISCHARGE_RATE\
                                                   ,SOC_MIN)
# repeat it for the rest of the records
for i in range(1, basic_p.shape[0]):
    basic_p.at[i,"SOC_start"] = basic_p.at[i-1,"SOC_end"]
    
    if basic_p.at[i,"surplus"] > 0:
        basic_p.at[i,"SOC_end"] = __builtins__.min(basic_p.at[i,"SOC_start"] + basic_p.at[i,"surplus"]\
                                                   ,basic_p.at[i,"SOC_start"] + CHARGE_RATE\
                                                   ,SOC_MAX)

    else:
        basic_p.at[i,"SOC_end"] = __builtins__.max(basic_p.at[i,"SOC_start"] - basic_p.at[i,"deficit"]\
                                                   ,basic_p.at[i,"SOC_start"] - DISCHARGE_RATE\
                                                   ,SOC_MIN)
#basic_p.tail(10)

# from the data previously calculated, we determine how much energy we used from our own storage and from the grid,
# furthermore how much energy we feed our system or the grid.
# last we calculate the energy price and the revenue. 
basic_p = basic_p.assign(dch = lambda x: np.where(x["SOC_start"] - x["SOC_end"] > 0, x["SOC_start"] - x["SOC_end"], 0),
                       ch    = lambda x: np.where(x["SOC_end"] - x["SOC_start"] > 0, x["SOC_end"] - x["SOC_start"], 0),
                       c_out = lambda x: x["deficit"]-x["dch"],
                       c_in  = lambda x: x["surplus"]-x["ch"],
                       price = lambda x: (x["c_out"]*x["price_eur_per_mwh"]-x["c_in"]*x["price_eur_per_mwh"])*0.00025) 

#basic_p.head(100)



In [214]:
simple_total_generation = basic_p["production_kwh"].sum()
simple_total_export = basic_p["c_in"].sum()
simple_self_consumption_rate = 1 - simple_total_export/simple_total_generation

simple_total_consumption = basic_p["consumption_kwh"].sum()
simple_total_import = basic_p["c_out"].sum()
simple_self_sufficiency_rate = 1 - simple_total_import/simple_total_consumption

simple_total_cost = basic_p[basic_p["price"]>0]["price"].sum()
simple_total_revenue = basic_p[basic_p["price"]<0]["price"].sum()*(-1)
simple_net_revenue = basic_p["price"].sum()*(-1)

print(f"Total generation: {simple_total_generation} kwh")
print(f"Total consumption: {simple_total_consumption} kwh")
print(f"Self consumption rate: {simple_self_consumption_rate}")
print(f"Self sufficiency rate: {simple_self_sufficiency_rate}")
print(f"Total cost: {simple_total_cost} €")
print(f"Total revenue: {simple_total_revenue} €")
print(f"Net revenue: {simple_net_revenue} €")

Total generation: 38.948577880859375 kwh
Total consumption: 172.95486450195312 kwh
Self consumption rate: 1.0
Self sufficiency rate: 0.2251950223253868
Total cost: 4.734774810046323 €
Total revenue: -0.0 €
Net revenue: -4.734774810046323 €


# Rule Based Strategy

If there is a deficit, we check whether the energy price reaches the current upper price threshold. If not we buy from the grid, if yes then we use the backup stored in BESS, if we can't cover demand from BESS, then power is bought from the grid.

For surplus, we load as much energy as we can into BESS, the remainder is sold to the grid if any.

In both cases we check whether the energy price is below the lower price threshold, if it is, then we charge the battery as much as we can from the grid.

The upper threshold is calculated from the upper (.1, .2, ... .9) percentile of the daily forcasted price data.
The lower threshold is calculated from the lower .05 percentile of the previous 15 days' price data.

both thresholds are recalculated every day.

In [14]:
#function for determining season
def get_season(timestamp):

    month = timestamp.month
    day = timestamp.day
    
    if (month == 12 and day >= 21) or (1 <= month <= 2) or (month == 3 and day <= 20):
        return "Winter"
    elif (month == 3 and day >= 21) or (4 <= month <= 5) or (month == 6 and day <= 20):
        return "Spring"
    elif (month == 6 and day >= 21) or (7 <= month <= 8) or (month == 9 and day <= 21):
        return "Summer"
    else:
        return "Autumn"
# spakr user defined function declaration
season_udf = udf(get_season, StringType())

In [191]:
percentiles = {"Winter":0.5,"Spring":0.1,"Summer":0.1,"Autumn":0.2}

In [215]:

rule_based = process.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
n_ut = 1
n_lt = 1

#first row
UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation="higher")
LT = 0

# comparing demand and prodcuction
# deficit side
if rule_based.at[0,"deficit"] > 0:
    
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[0,"price_eur_per_mwh"] < UT:
            ##P_B2L = 0
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < SOC_MAX):
               rule_based.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[0,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0
    
        # expensive
        else:
            ##P_G2B = 0

            rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"deficit"], rule_based.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"] - rule_based.at[0,"P_B2L"]
                
                
# surplus side            
else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
        rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"surplus"]\
                                                    ,SOC_MAX - rule_based.at[0,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"] - rule_based.at[0,"P_P2B"]
    
        if rule_based.at[0,"price_eur_per_mwh"] < LT:
                    rule_based.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[0,"P_P2B"], SOC_MAX - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
        ##else P_G2B = 0

# calculating prices and SOC
rule_based.at[0,"SOC_end"]    = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
rule_based.at[0,"buy_price"]  = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
rule_based.at[0,"sell_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025

# repeat it for each row
for i in range(1, rule_based.shape[0]):
    
    # setting SOC 
    rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
    
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based.shape[0]):
        UT = rule_based.iloc[i : i + 96]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation="higher")
    #if(n_ut % 96) & (n_ut >= 480):
        #UT = rule_based.iloc[i - 480 : i]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation="higher")
    
    if (n_lt % 96) & (n_lt >= 1440):
        LT = rule_based.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
    
    n_ut += 1
    n_lt += 1

    # comparing demand and production
    # deficit side
    if rule_based.at[i,"deficit"] > 0:
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based.at[i,"price_eur_per_mwh"] < UT:
            ##P_B2L = 0
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < SOC_MAX):
               rule_based.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[i,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0
    
        # expensive
        else:
            ##P_G2B = 0

            rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"] - rule_based.at[i,"P_B2L"]
            
                
    # surplus side            
    else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
        rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"surplus"]\
                                                    ,SOC_MAX - rule_based.at[i,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"] - rule_based.at[i,"P_P2B"]
    
        if rule_based.at[i,"price_eur_per_mwh"] < LT:
                    rule_based.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[i,"P_P2B"], SOC_MAX - rule_based.at[i,"SOC_start"] - rule_based.at[0,"P_P2B"])
        ##else P_G2B = 0
        
    #executing the calculations
    rule_based.at[i,"SOC_end"]    = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
    rule_based.at[i,"buy_price"]  = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
    rule_based.at[i,"sell_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0
LT=0


In [216]:
rule_total_generation = rule_based["production_kwh"].sum()
rule_total_export = rule_based["P_P2G"].sum()
rule_self_consumption_rate = 1 - rule_total_export/rule_total_generation

rule_total_consumption = rule_based["consumption_kwh"].sum()
rule_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
rule_self_sufficiency_rate = 1 - rule_total_import/rule_total_consumption

rule_total_cost = rule_based["buy_price"].sum()
rule_total_revenue = rule_based["sell_price"].sum()
rule_net_revenue = rule_total_revenue - rule_total_cost

print(f"Total generation: {rule_total_generation} kwh")
print(f"Total consumption: {rule_total_consumption} kwh")
print(f"Self consumption rate: {rule_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_self_sufficiency_rate}")
print(f"Total cost: {rule_total_cost} €")
print(f"Total revenue: {rule_total_revenue} €")
print(f"Net revenue: {rule_net_revenue} €")

Total generation: 38.948577880859375 kwh
Total consumption: 172.95486450195312 kwh
Self consumption rate: 1.0
Self sufficiency rate: 0.2251950223253868
Total cost: 4.5157267462929145 €
Total revenue: 0.0 €
Net revenue: -4.5157267462929145 €


# Rule Based Strategy with no LT

same strategy with one key difference: no lower threshold is used to buy cheaper energy.

In [140]:
rule_based_no_lt = process.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, x["production_kwh"] - x["consumption_kwh"], 0),                       deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       buy_price     = float(0.00000),
                       sell_price = float(0.0000))
n_ut = 1

#first row
UT = rule_based_no_lt.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based_no_lt.at[0,"timestamp"])], interpolation='higher')

## P_G2B = 0 always

# comparing demand and prodcuction
# deficit side
if rule_based_no_lt.at[0,"deficit"] > 0:
    
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[0,"price_eur_per_mwh"] < UT:
            ##P_B2L = 0
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"deficit"]

        # expensive
        else:
            ##P_G2B = 0
            
            rule_based_no_lt.at[0,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[0,"deficit"], rule_based_no_lt.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"deficit"] - rule_based_no_lt.at[0,"P_B2L"]
                
                
# surplus side            
else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"consumption_kwh"]
        rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[0,"surplus"]\
                                                    ,SOC_MAX - rule_based_no_lt.at[0,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"surplus"] - rule_based_no_lt.at[0,"P_P2B"]
    
# calculating prices and SOC
rule_based_no_lt.at[0,"SOC_end"]    = rule_based_no_lt.at[0,"SOC_start"] + rule_based_no_lt.at[0,"P_P2B"] + rule_based_no_lt.at[0,"P_G2B"] - rule_based_no_lt.at[0,"P_B2L"]
rule_based_no_lt.at[0,"buy_price"]  = (rule_based_no_lt.at[0,"P_G2B"] + rule_based_no_lt.at[0,"P_G2L"])*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025
rule_based_no_lt.at[0,"sell_price"] = rule_based_no_lt.at[0,"P_P2G"]*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025

# repeat it for each row
for i in range(1, rule_based_no_lt.shape[0]):
    
    # setting SOC 
    rule_based_no_lt.at[i,"SOC_start"] = rule_based_no_lt.at[i-1,"SOC_end"]
    
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based_no_lt.shape[0]):
        UT = rule_based_no_lt.iloc[i : i + 96]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based_no_lt.at[i,"timestamp"])], interpolation="higher")
        
    n_ut += 1

    # comparing demand and production
    # deficit side
    if rule_based_no_lt.at[i,"deficit"] > 0:
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"production_kwh"]
        
        # cheaper than UT
        if rule_based_no_lt.at[i,"price_eur_per_mwh"] < UT:
            ##P_B2L = 0
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"deficit"]

        # expensive
        else:
            ##P_G2B = 0

            rule_based_no_lt.at[i,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[i,"deficit"], rule_based_no_lt.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"deficit"] - rule_based_no_lt.at[i,"P_B2L"]
            
                
    # surplus side            
    else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"consumption_kwh"]
        rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[i,"surplus"]\
                                                    ,SOC_MAX - rule_based_no_lt.at[i,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"surplus"] - rule_based_no_lt.at[i,"P_P2B"]
        
    #executing the calculations
    rule_based_no_lt.at[i,"SOC_end"]    = rule_based_no_lt.at[i,"SOC_start"] + rule_based_no_lt.at[i,"P_P2B"] + rule_based_no_lt.at[i,"P_G2B"] - rule_based_no_lt.at[i,"P_B2L"]
    rule_based_no_lt.at[i,"buy_price"]  = (rule_based_no_lt.at[i,"P_G2B"] + rule_based_no_lt.at[i,"P_G2L"])*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
    rule_based_no_lt.at[i,"sell_price"] = rule_based_no_lt.at[i,"P_P2G"]*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0

In [141]:
rule_no_lt_total_generation = rule_based_no_lt["production_kwh"].sum()
rule_no_lt_total_export = rule_based_no_lt["P_P2G"].sum()
rule_no_lt_self_consumption_rate = 1 - rule_no_lt_total_export/rule_no_lt_total_generation

rule_no_lt_total_consumption = rule_based_no_lt["consumption_kwh"].sum()
rule_no_lt_total_import = rule_based_no_lt["P_G2L"].sum() + rule_based_no_lt["P_G2B"].sum()
rule_no_lt_self_sufficiency_rate = 1 - rule_no_lt_total_import/rule_no_lt_total_consumption

rule_no_lt_total_cost = rule_based_no_lt["buy_price"].sum()
rule_no_lt_total_revenue = rule_based_no_lt["sell_price"].sum()
rule_no_lt_net_revenue = rule_no_lt_total_revenue - rule_no_lt_total_cost

print(f"Total generation: {rule_no_lt_total_generation} kwh")
print(f"Total consumption: {rule_no_lt_total_consumption} kwh")
print(f"Self consumption rate: {rule_no_lt_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_no_lt_self_sufficiency_rate}")
print(f"Total cost: {rule_no_lt_total_cost} €")
print(f"Total revenue: {rule_no_lt_total_revenue} €")
print(f"Net revenue: {rule_no_lt_net_revenue} €")

Total generation: 21045.080078125 kwh
Total consumption: 19999.2734375 kwh
Self consumption rate: 0.6593267945866693
Self sufficiency rate: 0.6938044216025108
Total cost: 286.38550341514764 €
Total revenue: 368.85769091519904 €
Net revenue: 82.4721875000514 €


# Wild idea

In [151]:
battery_threshold = 20

**lt**

In [152]:

rule_based = process.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
n_ut = 1
n_lt = 1

#first row
UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation="higher")
LT = 0

# comparing demand and prodcuction
# deficit side
if rule_based.at[0,"deficit"] > 0:
    
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
        
        # cheaper than UT and charge state is low
        if (rule_based.at[0,"SOC_start"] <= battery_threshold) & (rule_based.at[0,"price_eur_per_mwh"] < UT):
            ##P_B2L = 0
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < SOC_MAX):
               rule_based.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[0,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0
    
        # expensive or charge state is acceptable
        else:
            #cheaper than LT and there is space
            if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < SOC_MAX):
               rule_based.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[0,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0
            
            rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"deficit"], rule_based.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"] - rule_based.at[0,"P_B2L"]
                
                
# surplus side            
else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
        rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"surplus"]\
                                                    ,SOC_MAX - rule_based.at[0,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"] - rule_based.at[0,"P_P2B"]
    
        if rule_based.at[0,"price_eur_per_mwh"] < LT:
                    rule_based.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[0,"P_P2B"], SOC_MAX - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
        ##else P_G2B = 0

# calculating prices and SOC
rule_based.at[0,"SOC_end"]    = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
rule_based.at[0,"buy_price"]  = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
rule_based.at[0,"sell_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025

# repeat it for each row
for i in range(1, rule_based.shape[0]):
    
    # setting SOC 
    rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
    
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based.shape[0]):
        UT = rule_based.iloc[i : i + 96]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation="higher")
    #if(n_ut % 96) & (n_ut >= 480):
        #UT = rule_based.iloc[i - 480 : i]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation="higher")
    
    if (n_lt % 96) & (n_lt >= 1440):
        LT = rule_based.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
    
    n_ut += 1
    n_lt += 1

    # comparing demand and production
    # deficit side
    if rule_based.at[i,"deficit"] > 0:
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
        
        # cheaper than UT and charge state is low
        if (rule_based.at[i,"SOC_start"] <= battery_threshold) & (rule_based.at[i,"price_eur_per_mwh"] < UT):
            ##P_B2L = 0
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"]

            #cheaper than LT and there is space
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < SOC_MAX):
               rule_based.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[i,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0
    
        # expensive or charge state is acceptable
        else:
            if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < SOC_MAX):
               rule_based.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[i,"SOC_start"],CHARGE_RATE])
            ##else P_G2B = 0

            rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"] - rule_based.at[i,"P_B2L"]
            
                
    # surplus side            
    else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
        rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"surplus"]\
                                                    ,SOC_MAX - rule_based.at[i,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"] - rule_based.at[i,"P_P2B"]
    
        if rule_based.at[i,"price_eur_per_mwh"] < LT:
                    rule_based.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[i,"P_P2B"], SOC_MAX - rule_based.at[i,"SOC_start"] - rule_based.at[0,"P_P2B"])
        ##else P_G2B = 0
        
    #executing the calculations
    rule_based.at[i,"SOC_end"]    = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
    rule_based.at[i,"buy_price"]  = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
    rule_based.at[i,"sell_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0
LT=0

In [153]:
rule_total_generation = rule_based["production_kwh"].sum()
rule_total_export = rule_based["P_P2G"].sum()
rule_self_consumption_rate = 1 - rule_total_export/rule_total_generation

rule_total_consumption = rule_based["consumption_kwh"].sum()
rule_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
rule_self_sufficiency_rate = 1 - rule_total_import/rule_total_consumption

rule_total_cost = rule_based["buy_price"].sum()
rule_total_revenue = rule_based["sell_price"].sum()
rule_net_revenue = rule_total_revenue - rule_total_cost

print(f"Total generation: {rule_total_generation} kwh")
print(f"Total consumption: {rule_total_consumption} kwh")
print(f"Self consumption rate: {rule_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_self_sufficiency_rate}")
print(f"Total cost: {rule_total_cost} €")
print(f"Total revenue: {rule_total_revenue} €")
print(f"Net revenue: {rule_net_revenue} €")

Total generation: 21045.080078125 kwh
Total consumption: 19999.2734375 kwh
Self consumption rate: 0.6115946550432424
Self sufficiency rate: 0.6435762619580658
Total cost: 256.7599632721675 €
Total revenue: 387.8855000085685 €
Net revenue: 131.125536736401 €


**no lt**

In [149]:
rule_based_no_lt = process.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, x["production_kwh"] - x["consumption_kwh"], 0),                       deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, x['consumption_kwh'] - x['production_kwh'], 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       buy_price     = float(0.00000),
                       sell_price = float(0.0000))
n_ut = 1

#first row
UT = rule_based_no_lt.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based_no_lt.at[0,"timestamp"])], interpolation='higher')

## P_G2B = 0 always

# comparing demand and prodcuction
# deficit side
if rule_based_no_lt.at[0,"deficit"] > 0:
    
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"production_kwh"]

        # cheaper than UT and charge state is low
        if (rule_based_no_lt.at[0,"SOC_start"] <= battery_threshold) & (rule_based_no_lt.at[0,"price_eur_per_mwh"] < UT):
        
            ##P_B2L = 0
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"deficit"]

        # expensive or charge state is acceptable
        else:
            ##P_G2B = 0
            
            rule_based_no_lt.at[0,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[0,"deficit"], rule_based_no_lt.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based_no_lt.at[0,"P_G2L"] = rule_based_no_lt.at[0,"deficit"] - rule_based_no_lt.at[0,"P_B2L"]
                
                
# surplus side            
else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based_no_lt.at[0,"P_P2L"] = rule_based_no_lt.at[0,"consumption_kwh"]
        rule_based_no_lt.at[0,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[0,"surplus"]\
                                                    ,SOC_MAX - rule_based_no_lt.at[0,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based_no_lt.at[0,"P_P2G"] = rule_based_no_lt.at[0,"surplus"] - rule_based_no_lt.at[0,"P_P2B"]
    
# calculating prices and SOC
rule_based_no_lt.at[0,"SOC_end"]    = rule_based_no_lt.at[0,"SOC_start"] + rule_based_no_lt.at[0,"P_P2B"] + rule_based_no_lt.at[0,"P_G2B"] - rule_based_no_lt.at[0,"P_B2L"]
rule_based_no_lt.at[0,"buy_price"]  = (rule_based_no_lt.at[0,"P_G2B"] + rule_based_no_lt.at[0,"P_G2L"])*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025
rule_based_no_lt.at[0,"sell_price"] = rule_based_no_lt.at[0,"P_P2G"]*rule_based_no_lt.at[0,"price_eur_per_mwh"]*0.00025

# repeat it for each row
for i in range(1, rule_based_no_lt.shape[0]):
    
    # setting SOC 
    rule_based_no_lt.at[i,"SOC_start"] = rule_based_no_lt.at[i-1,"SOC_end"]
    
    # calculating thresholds
    if (n_ut % 96 == 0) & (n_ut != rule_based_no_lt.shape[0]):
        UT = rule_based_no_lt.iloc[i : i + 96]["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based_no_lt.at[i,"timestamp"])], interpolation="higher")
        
    n_ut += 1

    # comparing demand and production
    # deficit side
    if rule_based_no_lt.at[i,"deficit"] > 0:
        ##P_P2B = 0
        ##P_P2G = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"production_kwh"]
        

        if (rule_based_no_lt.at[i,"SOC_start"] <= battery_threshold) & (rule_based_no_lt.at[i,"price_eur_per_mwh"] < UT):
            ##P_B2L = 0
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"deficit"]

        else:
            ##P_G2B = 0

            rule_based_no_lt.at[i,"P_B2L"] = __builtins__.min([rule_based_no_lt.at[i,"deficit"], rule_based_no_lt.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
            rule_based_no_lt.at[i,"P_G2L"] = rule_based_no_lt.at[i,"deficit"] - rule_based_no_lt.at[i,"P_B2L"]
            
                
    # surplus side            
    else:
        ##P_G2L = 0
        ##P_B2L = 0
        rule_based_no_lt.at[i,"P_P2L"] = rule_based_no_lt.at[i,"consumption_kwh"]
        rule_based_no_lt.at[i,"P_P2B"] = __builtins__.min(rule_based_no_lt.at[i,"surplus"]\
                                                    ,SOC_MAX - rule_based_no_lt.at[i,"SOC_start"]\
                                                    ,CHARGE_RATE)
        rule_based_no_lt.at[i,"P_P2G"] = rule_based_no_lt.at[i,"surplus"] - rule_based_no_lt.at[i,"P_P2B"]
        
    #executing the calculations
    rule_based_no_lt.at[i,"SOC_end"]    = rule_based_no_lt.at[i,"SOC_start"] + rule_based_no_lt.at[i,"P_P2B"] + rule_based_no_lt.at[i,"P_G2B"] - rule_based_no_lt.at[i,"P_B2L"]
    rule_based_no_lt.at[i,"buy_price"]  = (rule_based_no_lt.at[i,"P_G2B"] + rule_based_no_lt.at[i,"P_G2L"])*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
    rule_based_no_lt.at[i,"sell_price"] = rule_based_no_lt.at[i,"P_P2G"]*rule_based_no_lt.at[i,"price_eur_per_mwh"]*0.00025
            
UT=0

In [150]:
rule_no_lt_total_generation = rule_based_no_lt["production_kwh"].sum()
rule_no_lt_total_export = rule_based_no_lt["P_P2G"].sum()
rule_no_lt_self_consumption_rate = 1 - rule_no_lt_total_export/rule_no_lt_total_generation

rule_no_lt_total_consumption = rule_based_no_lt["consumption_kwh"].sum()
rule_no_lt_total_import = rule_based_no_lt["P_G2L"].sum() + rule_based_no_lt["P_G2B"].sum()
rule_no_lt_self_sufficiency_rate = 1 - rule_no_lt_total_import/rule_no_lt_total_consumption

rule_no_lt_total_cost = rule_based_no_lt["buy_price"].sum()
rule_no_lt_total_revenue = rule_based_no_lt["sell_price"].sum()
rule_no_lt_net_revenue = rule_no_lt_total_revenue - rule_no_lt_total_cost

print(f"Total generation: {rule_no_lt_total_generation} kwh")
print(f"Total consumption: {rule_no_lt_total_consumption} kwh")
print(f"Self consumption rate: {rule_no_lt_self_consumption_rate}")
print(f"Self sufficiency rate: {rule_no_lt_self_sufficiency_rate}")
print(f"Total cost: {rule_no_lt_total_cost} €")
print(f"Total revenue: {rule_no_lt_total_revenue} €")
print(f"Net revenue: {rule_no_lt_net_revenue} €")

Total generation: 21045.080078125 kwh
Total consumption: 19999.2734375 kwh
Self consumption rate: 0.6702966179594438
Self sufficiency rate: 0.705347881520912
Total cost: 276.2587286296736 €
Total revenue: 356.4534057365468 €
Net revenue: 80.1946771068732 €


# Experimenting for rule based hyper parametering

getting seasonal data:

In [96]:
seasonal_data = production.withColumn("season",season_udf(col("timestamp"))).withColumn("week", weekofyear(col("timestamp"))).withColumn("year",year(col("timestamp")))
avg_data= seasonal_data.groupBy("season").agg(avg("production_kwh").alias("avg_production"))
seasonal_data = seasonal_data.join(avg_data,"season","inner")\
                             .groupBy("season","week").agg(avg("production_kwh").alias("weekly_avg"),max("avg_production").alias("seasonal_avg"),max("year").alias("year"))\
                             .withColumn("diff",abs(col("seasonal_avg")-col("weekly_avg")))
window_spec = Window.partitionBy("season").orderBy(col("diff"))

closest_weeks = seasonal_data.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

closest_weeks.select("season","year","week","seasonal_avg","weekly_avg","diff").show()


+------+----+----+-------------------+-------------------+--------------------+
|season|year|week|       seasonal_avg|         weekly_avg|                diff|
+------+----+----+-------------------+-------------------+--------------------+
|Autumn|2023|  45|0.02909188788646503|0.02687262195644717|0.002219265930017...|
|Spring|2023|  17|0.07578741795865708|0.07697618300509501|0.001188765046437...|
|Summer|2023|  36| 0.0790577951227563|0.07917462784605875|1.168327233024429...|
|Winter|2023|   5|0.03729964863615653|0.03529165510024738| 0.00200799353590915|
+------+----+----+-------------------+-------------------+--------------------+



extracting the chosen weeks according to the results:

In [97]:
test_data = spark.createDataFrame(process).withColumn("season",season_udf(col("timestamp")))\
                      .withColumn("week", weekofyear(col("timestamp")))\
                      .withColumn("year",year(col("timestamp")))\
                   .filter((col("year") == 2023) & ((col("week") == 5) | (col("week") == 17) | (col("week") == 36) | (col("week") == 45)))
test_data = test_data.toPandas()
test_data.head()



24/07/01 17:27:18 WARN TaskSetManager: Stage 184 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,season,week,year
0,2023-01-30 00:00:00,0.18221,0.0,79.129997,Winter,5,2023
1,2023-01-30 00:15:00,0.18426,0.0,79.129997,Winter,5,2023
2,2023-01-30 00:30:00,0.18654,0.0,79.129997,Winter,5,2023
3,2023-01-30 00:45:00,0.18693,0.0,79.129997,Winter,5,2023
4,2023-01-30 01:00:00,0.18698,0.0,52.790001,Winter,5,2023


**Second method**

In [122]:
test_data = spark.createDataFrame(process).withColumn("season", season_udf(col("timestamp")))

testing the different percentiles for each season:

In [144]:
import numpy as np
values = [__builtins__.round(i*0.1,1) for i in range(1,8)]
print(values)
cols = ["percentile", "self_consumption_rate","self_sufficiency_rate","total_cost","total_revenue","net_revenue"]
test_percentiles = {"Winter":0.0,"Spring":0.0,"Summer":0.0,"Autumn":0.0}

[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]


**Winter**

In [145]:
winter = []
winter_data = test_data.filter(col("season") == "Winter").toPandas()
c = 0
for v in values:
    test_percentiles["Winter"] = v
    test_percentiles["Spring"] = 0
    test_percentiles["Summer"] = 0
    test_percentiles["Autmun"] = 0

    # rule based strategy
    rule_based_test = winter_data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))

    rule_based_test.reset_index(drop=True, inplace=True)
    
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based_test.iloc[0 : 96]['price_eur_per_mwh'].quantile(test_percentiles[get_season(rule_based_test.at[0,"timestamp"])], interpolation="higher")
    LT = 0
    
    # comparing demand and prodcuction
    # deficit side
    if rule_based_test.at[0,"deficit"] > 0:
        
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[0,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[0,"price_eur_per_mwh"] < LT) & (rule_based_test.at[0,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[0,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[0,"P_B2L"] = __builtins__.min([rule_based_test.at[0,"deficit"], rule_based_test.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"] - rule_based_test.at[0,"P_B2L"]
                    
                    
    # surplus side            
    else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"consumption_kwh"]
            rule_based_test.at[0,"P_P2B"] = __builtins__.min(rule_based_test.at[0,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[0,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[0,"P_P2G"] = rule_based_test.at[0,"surplus"] - rule_based_test.at[0,"P_P2B"]
        
            if rule_based_test.at[0,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[0,"P_P2B"], SOC_MAX - rule_based_test.at[0,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
    
    # calculating prices and SOC
    rule_based_test.at[0,"SOC_end"]    = rule_based_test.at[0,"SOC_start"] + rule_based_test.at[0,"P_P2B"] + rule_based_test.at[0,"P_G2B"] - rule_based_test.at[0,"P_B2L"]
    rule_based_test.at[0,"buy_price"]  = (rule_based_test.at[0,"P_G2B"] + rule_based_test.at[0,"P_G2L"])*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    rule_based_test.at[0,"sell_price"] = rule_based_test.at[0,"P_P2G"]*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    
    # repeat it for each row
    for i in range(1, rule_based_test.shape[0]):
        
        # setting SOC 
        rule_based_test.at[i,"SOC_start"] = rule_based_test.at[i-1,"SOC_end"]
        
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based_test.shape[0]):
            UT = rule_based_test.iloc[i : i + 96]["price_eur_per_mwh"].quantile(test_percentiles[get_season(rule_based_test.at[i,"timestamp"])], interpolation="higher")
        
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based_test.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
        
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # deficit side
        if rule_based_test.at[i,"deficit"] > 0:
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[i,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[i,"price_eur_per_mwh"] < LT) & (rule_based_test.at[i,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[i,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[i,"P_B2L"] = __builtins__.min([rule_based_test.at[i,"deficit"], rule_based_test.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"] - rule_based_test.at[i,"P_B2L"]
                
                    
        # surplus side            
        else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"consumption_kwh"]
            rule_based_test.at[i,"P_P2B"] = __builtins__.min(rule_based_test.at[i,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[i,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[i,"P_P2G"] = rule_based_test.at[i,"surplus"] - rule_based_test.at[i,"P_P2B"]
        
            if rule_based_test.at[i,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[i,"P_P2B"], SOC_MAX - rule_based_test.at[i,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
            
        #executing the calculations
        rule_based_test.at[i,"SOC_end"]    = rule_based_test.at[i,"SOC_start"] + rule_based_test.at[i,"P_P2B"] + rule_based_test.at[i,"P_G2B"] - rule_based_test.at[i,"P_B2L"]
        rule_based_test.at[i,"buy_price"]  = (rule_based_test.at[i,"P_G2B"] + rule_based_test.at[i,"P_G2L"])*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
        rule_based_test.at[i,"sell_price"] = rule_based_test.at[i,"P_P2G"]*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0

    c += 1
    print(c)
    

    r_total_generation = rule_based_test["production_kwh"].sum()
    r_total_export = rule_based_test["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based_test["consumption_kwh"].sum()
    r_total_import = rule_based_test["P_G2L"].sum() + rule_based_test["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based_test["buy_price"].sum()
    r_total_revenue = rule_based_test["sell_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    winter.append((v, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

winter_df = spark.createDataFrame(winter, cols)
winter_df.show()

24/07/01 17:50:44 WARN TaskSetManager: Stage 329 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


1
2
3
4
5
6
7
+----------+---------------------+---------------------+------------------+------------------+-------------------+
|percentile|self_consumption_rate|self_sufficiency_rate|        total_cost|     total_revenue|        net_revenue|
+----------+---------------------+---------------------+------------------+------------------+-------------------+
|       0.1|   0.8716133460690135|   0.4861707360181433|121.09241250128282|15.977188026475828|  -105.115224474807|
|       0.2|   0.8606229203846374|  0.48004046808658407|121.87141070465029|17.730999142255506|-104.14041156239477|
|       0.3|   0.8558489231593586|  0.47737761562427383|121.70244509126286|18.500477942937028|-103.20196714832583|
|       0.4|   0.8529212669294545|  0.47574461999551265|121.78389918750244|18.824702020821274|-102.95919716668116|
|       0.5|   0.8466367184364636|  0.47173644508674817|122.84969906488551| 19.52898701032774|-103.32071205455777|
|       0.6|   0.8360089127334068|  0.46456691700450814| 125.56499

**Spring**

In [84]:
spring = []
spring_data = test_data.filter(col("season") == "Spring").toPandas()
c = 0
for v in values:
    test_percentiles["Winter"] = 0
    test_percentiles["Spring"] = v
    test_percentiles["Summer"] = 0
    test_percentiles["Autmun"] = 0

    # rule based strategy
    rule_based_test = spring_data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
    rule_based_test.reset_index(drop=True, inplace=True)
    
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based_test.iloc[0 : 96]['price_eur_per_mwh'].quantile(test_percentiles[get_season(rule_based_test.at[0,"timestamp"])], interpolation="higher")
    LT = 0
    
    # comparing demand and prodcuction
    # deficit side
    if rule_based_test.at[0,"deficit"] > 0:
        
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[0,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[0,"price_eur_per_mwh"] < LT) & (rule_based_test.at[0,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[0,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[0,"P_B2L"] = __builtins__.min([rule_based_test.at[0,"deficit"], rule_based_test.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"] - rule_based_test.at[0,"P_B2L"]
                    
                    
    # surplus side            
    else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"consumption_kwh"]
            rule_based_test.at[0,"P_P2B"] = __builtins__.min(rule_based_test.at[0,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[0,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[0,"P_P2G"] = rule_based_test.at[0,"surplus"] - rule_based_test.at[0,"P_P2B"]
        
            if rule_based_test.at[0,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[0,"P_P2B"], SOC_MAX - rule_based_test.at[0,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
    
    # calculating prices and SOC
    rule_based_test.at[0,"SOC_end"]    = rule_based_test.at[0,"SOC_start"] + rule_based_test.at[0,"P_P2B"] + rule_based_test.at[0,"P_G2B"] - rule_based_test.at[0,"P_B2L"]
    rule_based_test.at[0,"buy_price"]  = (rule_based_test.at[0,"P_G2B"] + rule_based_test.at[0,"P_G2L"])*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    rule_based_test.at[0,"sell_price"] = rule_based_test.at[0,"P_P2G"]*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    
    # repeat it for each row
    for i in range(1, rule_based_test.shape[0]):
        
        # setting SOC 
        rule_based_test.at[i,"SOC_start"] = rule_based_test.at[i-1,"SOC_end"]
        
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based_test.shape[0]):
            UT = rule_based_test.iloc[i : i + 96]["price_eur_per_mwh"].quantile(test_percentiles[get_season(rule_based_test.at[i,"timestamp"])], interpolation="higher")
        
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based_test.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
        
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # deficit side
        if rule_based_test.at[i,"deficit"] > 0:
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[i,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[i,"price_eur_per_mwh"] < LT) & (rule_based_test.at[i,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[i,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[i,"P_B2L"] = __builtins__.min([rule_based_test.at[i,"deficit"], rule_based_test.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"] - rule_based_test.at[i,"P_B2L"]
                
                    
        # surplus side            
        else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"consumption_kwh"]
            rule_based_test.at[i,"P_P2B"] = __builtins__.min(rule_based_test.at[i,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[i,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[i,"P_P2G"] = rule_based_test.at[i,"surplus"] - rule_based_test.at[i,"P_P2B"]
        
            if rule_based_test.at[i,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[i,"P_P2B"], SOC_MAX - rule_based_test.at[i,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
            
        #executing the calculations
        rule_based_test.at[i,"SOC_end"]    = rule_based_test.at[i,"SOC_start"] + rule_based_test.at[i,"P_P2B"] + rule_based_test.at[i,"P_G2B"] - rule_based_test.at[i,"P_B2L"]
        rule_based_test.at[i,"buy_price"]  = (rule_based_test.at[i,"P_G2B"] + rule_based_test.at[i,"P_G2L"])*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
        rule_based_test.at[i,"sell_price"] = rule_based_test.at[i,"P_P2G"]*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0

    c += 1
    print(c)

    r_total_generation = rule_based_test["production_kwh"].sum()
    r_total_export = rule_based_test["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based_test["consumption_kwh"].sum()
    r_total_import = rule_based_test["P_G2L"].sum() + rule_based_test["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based_test["buy_price"].sum()
    r_total_revenue = rule_based_test["sell_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    spring.append((v, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

spring_df = spark.createDataFrame(spring, cols)
spring_df.show()

24/07/01 17:06:14 WARN TaskSetManager: Stage 137 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


+----------+---------------------+---------------------+------------------+------------------+-------------------+
|percentile|self_consumption_rate|self_sufficiency_rate|        total_cost|     total_revenue|        net_revenue|
+----------+---------------------+---------------------+------------------+------------------+-------------------+
|       0.1|   0.8497148468178544|    0.643619797371138|44.449363112390955|10.861308313312389| -33.58805479907856|
|       0.2|   0.8454389738740054|   0.6403302387232593|44.970772707131594|11.585731694297936| -33.38504101283366|
|       0.3|   0.8377662535116488|   0.6344273824054512| 46.64556579219454|12.841242877752565|-33.804322914441975|
|       0.4|   0.8325235767513991|   0.6303940322799474| 48.35997994088293|13.556409345169879|-34.803570595713055|
|       0.5|     0.82122733969894|   0.6215973499639832|51.999890443859485|15.130120797666486|   -36.869769646193|
+----------+---------------------+---------------------+------------------+-----

**Autumn**

In [86]:
autumn = []
autumn_data = test_data.filter(col("season") == "Autumn").toPandas()
c = 0
for v in values:
    test_percentiles["Winter"] = 0
    test_percentiles["Spring"] = 0
    test_percentiles["Summer"] = 0
    test_percentiles["Autmun"] = v

    # rule based strategy
    rule_based_test = autumn_data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
    rule_based_test.reset_index(drop=True, inplace=True)
    
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based_test.iloc[0 : 96]['price_eur_per_mwh'].quantile(test_percentiles[get_season(rule_based_test.at[0,"timestamp"])], interpolation="higher")
    LT = 0
    
    # comparing demand and prodcuction
    # deficit side
    if rule_based_test.at[0,"deficit"] > 0:
        
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[0,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[0,"price_eur_per_mwh"] < LT) & (rule_based_test.at[0,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[0,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[0,"P_B2L"] = __builtins__.min([rule_based_test.at[0,"deficit"], rule_based_test.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"] - rule_based_test.at[0,"P_B2L"]
                    
                    
    # surplus side            
    else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"consumption_kwh"]
            rule_based_test.at[0,"P_P2B"] = __builtins__.min(rule_based_test.at[0,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[0,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[0,"P_P2G"] = rule_based_test.at[0,"surplus"] - rule_based_test.at[0,"P_P2B"]
        
            if rule_based_test.at[0,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[0,"P_P2B"], SOC_MAX - rule_based_test.at[0,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
    
    # calculating prices and SOC
    rule_based_test.at[0,"SOC_end"]    = rule_based_test.at[0,"SOC_start"] + rule_based_test.at[0,"P_P2B"] + rule_based_test.at[0,"P_G2B"] - rule_based_test.at[0,"P_B2L"]
    rule_based_test.at[0,"buy_price"]  = (rule_based_test.at[0,"P_G2B"] + rule_based_test.at[0,"P_G2L"])*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    rule_based_test.at[0,"sell_price"] = rule_based_test.at[0,"P_P2G"]*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    
    # repeat it for each row
    for i in range(1, rule_based_test.shape[0]):
        
        # setting SOC 
        rule_based_test.at[i,"SOC_start"] = rule_based_test.at[i-1,"SOC_end"]
        
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based_test.shape[0]):
            UT = rule_based_test.iloc[i : i + 96]["price_eur_per_mwh"].quantile(test_percentiles[get_season(rule_based_test.at[i,"timestamp"])], interpolation="higher")
        
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based_test.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
        
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # deficit side
        if rule_based_test.at[i,"deficit"] > 0:
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[i,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[i,"price_eur_per_mwh"] < LT) & (rule_based_test.at[i,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[i,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[i,"P_B2L"] = __builtins__.min([rule_based_test.at[i,"deficit"], rule_based_test.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"] - rule_based_test.at[i,"P_B2L"]
                
                    
        # surplus side            
        else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"consumption_kwh"]
            rule_based_test.at[i,"P_P2B"] = __builtins__.min(rule_based_test.at[i,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[i,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[i,"P_P2G"] = rule_based_test.at[i,"surplus"] - rule_based_test.at[i,"P_P2B"]
        
            if rule_based_test.at[i,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[i,"P_P2B"], SOC_MAX - rule_based_test.at[i,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
            
        #executing the calculations
        rule_based_test.at[i,"SOC_end"]    = rule_based_test.at[i,"SOC_start"] + rule_based_test.at[i,"P_P2B"] + rule_based_test.at[i,"P_G2B"] - rule_based_test.at[i,"P_B2L"]
        rule_based_test.at[i,"buy_price"]  = (rule_based_test.at[i,"P_G2B"] + rule_based_test.at[i,"P_G2L"])*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
        rule_based_test.at[i,"sell_price"] = rule_based_test.at[i,"P_P2G"]*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0

    c += 1
    print(c)

    r_total_generation = rule_based_test["production_kwh"].sum()
    r_total_export = rule_based_test["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based_test["consumption_kwh"].sum()
    r_total_import = rule_based_test["P_G2L"].sum() + rule_based_test["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based_test["buy_price"].sum()
    r_total_revenue = rule_based_test["sell_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    autumn.append((v, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

autumn_df = spark.createDataFrame(autumn, cols)
autumn_df.show()

24/07/01 17:09:43 WARN TaskSetManager: Stage 140 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


+----------+---------------------+---------------------+-----------------+------------------+-------------------+
|percentile|self_consumption_rate|self_sufficiency_rate|       total_cost|     total_revenue|        net_revenue|
+----------+---------------------+---------------------+-----------------+------------------+-------------------+
|       0.1|   0.9538131448874705|  0.23335875504106618|204.9118615136917|1.0096240731271997|-203.90223744056448|
|       0.2|   0.9538131448874705|  0.23335875504106618|204.9118615136917|1.0096240731271997|-203.90223744056448|
|       0.3|   0.9538131448874705|  0.23335875504106618|204.9118615136917|1.0096240731271997|-203.90223744056448|
|       0.4|   0.9538131448874705|  0.23335875504106618|204.9118615136917|1.0096240731271997|-203.90223744056448|
|       0.5|   0.9538131448874705|  0.23335875504106618|204.9118615136917|1.0096240731271997|-203.90223744056448|
+----------+---------------------+---------------------+-----------------+--------------

**Summer**

In [88]:
summer = []
summer_data = test_data.filter(col("season") == "Summer").toPandas()
c = 0
for v in values:
    test_percentiles["Winter"] = 0
    test_percentiles["Spring"] = 0
    test_percentiles["Summer"] = v
    test_percentiles["Autmun"] = 0

    # rule based strategy
    rule_based_test = summer_data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
    rule_based_test.reset_index(drop=True, inplace=True)
    
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based_test.iloc[0 : 96]['price_eur_per_mwh'].quantile(test_percentiles[get_season(rule_based_test.at[0,"timestamp"])], interpolation="higher")
    LT = 0
    
    # comparing demand and prodcuction
    # deficit side
    if rule_based_test.at[0,"deficit"] > 0:
        
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[0,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[0,"price_eur_per_mwh"] < LT) & (rule_based_test.at[0,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[0,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[0,"P_B2L"] = __builtins__.min([rule_based_test.at[0,"deficit"], rule_based_test.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[0,"P_G2L"] = rule_based_test.at[0,"deficit"] - rule_based_test.at[0,"P_B2L"]
                    
                    
    # surplus side            
    else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[0,"P_P2L"] = rule_based_test.at[0,"consumption_kwh"]
            rule_based_test.at[0,"P_P2B"] = __builtins__.min(rule_based_test.at[0,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[0,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[0,"P_P2G"] = rule_based_test.at[0,"surplus"] - rule_based_test.at[0,"P_P2B"]
        
            if rule_based_test.at[0,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[0,"P_P2B"], SOC_MAX - rule_based_test.at[0,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
    
    # calculating prices and SOC
    rule_based_test.at[0,"SOC_end"]    = rule_based_test.at[0,"SOC_start"] + rule_based_test.at[0,"P_P2B"] + rule_based_test.at[0,"P_G2B"] - rule_based_test.at[0,"P_B2L"]
    rule_based_test.at[0,"buy_price"]  = (rule_based_test.at[0,"P_G2B"] + rule_based_test.at[0,"P_G2L"])*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    rule_based_test.at[0,"sell_price"] = rule_based_test.at[0,"P_P2G"]*rule_based_test.at[0,"price_eur_per_mwh"]*0.00025
    
    # repeat it for each row
    for i in range(1, rule_based_test.shape[0]):
        
        # setting SOC 
        rule_based_test.at[i,"SOC_start"] = rule_based_test.at[i-1,"SOC_end"]
        
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based_test.shape[0]):
            UT = rule_based_test.iloc[i : i + 96]["price_eur_per_mwh"].quantile(test_percentiles[get_season(rule_based_test.at[i,"timestamp"])], interpolation="higher")
        
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based_test.iloc[i - 1440 : i]["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
        
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # deficit side
        if rule_based_test.at[i,"deficit"] > 0:
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based_test.at[i,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based_test.at[i,"price_eur_per_mwh"] < LT) & (rule_based_test.at[i,"SOC_start"] < SOC_MAX):
                   rule_based_test.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based_test.at[i,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based_test.at[i,"P_B2L"] = __builtins__.min([rule_based_test.at[i,"deficit"], rule_based_test.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based_test.at[i,"P_G2L"] = rule_based_test.at[i,"deficit"] - rule_based_test.at[i,"P_B2L"]
                
                    
        # surplus side            
        else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based_test.at[i,"P_P2L"] = rule_based_test.at[i,"consumption_kwh"]
            rule_based_test.at[i,"P_P2B"] = __builtins__.min(rule_based_test.at[i,"surplus"]\
                                                        ,SOC_MAX - rule_based_test.at[i,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based_test.at[i,"P_P2G"] = rule_based_test.at[i,"surplus"] - rule_based_test.at[i,"P_P2B"]
        
            if rule_based_test.at[i,"price_eur_per_mwh"] < LT:
                        rule_based_test.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based_test.at[i,"P_P2B"], SOC_MAX - rule_based_test.at[i,"SOC_start"] - rule_based_test.at[0,"P_P2B"])
            ##else P_G2B = 0
            
        #executing the calculations
        rule_based_test.at[i,"SOC_end"]    = rule_based_test.at[i,"SOC_start"] + rule_based_test.at[i,"P_P2B"] + rule_based_test.at[i,"P_G2B"] - rule_based_test.at[i,"P_B2L"]
        rule_based_test.at[i,"buy_price"]  = (rule_based_test.at[i,"P_G2B"] + rule_based_test.at[i,"P_G2L"])*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
        rule_based_test.at[i,"sell_price"] = rule_based_test.at[i,"P_P2G"]*rule_based_test.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0

    c += 1
    print(c)

    r_total_generation = rule_based_test["production_kwh"].sum()
    r_total_export = rule_based_test["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based_test["consumption_kwh"].sum()
    r_total_import = rule_based_test["P_G2L"].sum() + rule_based_test["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based_test["buy_price"].sum()
    r_total_revenue = rule_based_test["sell_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    summer.append((v, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

summer_df = spark.createDataFrame(summer, cols)
summer_df.show()

24/07/01 17:16:07 WARN TaskSetManager: Stage 146 contains a task of very large size (1338 KiB). The maximum recommended task size is 1000 KiB.


+----------+---------------------+---------------------+------------------+------------------+------------------+
|percentile|self_consumption_rate|self_sufficiency_rate|        total_cost|     total_revenue|       net_revenue|
+----------+---------------------+---------------------+------------------+------------------+------------------+
|       0.1|   0.7335788579346729|   0.7959263300716142|24.130918673589715|55.456806312882065| 31.32588763929235|
|       0.2|   0.7143958627688358|   0.7739847262224018|30.232963204520686| 61.84100073379534|31.608037529274654|
|       0.3|   0.6891529450489046|   0.7439435118991164|38.497398341304354| 69.70668317749069|31.209284836186335|
|       0.4|   0.6729772000957523|    0.725251616981341| 43.81355589232081|  74.1550240963187| 30.34146820399789|
|       0.5|   0.6410136099392437|    0.687974659695641|54.364678388615836|  81.8340738724183|27.469395483802465|
+----------+---------------------+---------------------+------------------+-------------

# testing the different percentile variations:

In [98]:
values = [0.1,0.5]
variations = list(itertools.product(values,repeat=4))


In [99]:
dataset = []
c = 0
columns = ["percentiles", "self_consumption_rate","self_sufficiency_rate","total_cost","total_revenue","net_revenue"]
for v in variations:
    percentiles['Winter'] = v[0]
    percentiles['Spring'] = v[1]
    percentiles['Summer'] = v[2]
    percentiles['Autumn'] = v[3]
    rule_based = test_data.assign(surplus = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh']), 0),
                            deficit  = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh']), 0),
                       SOC_start     = float(0.00000),
                       SOC_end       = float(0.00000),
                       P_P2L         = float(0.00000),
                       P_P2B         = float(0.00000),
                       P_P2G         = float(0.00000),
                       P_G2B         = float(0.00000),
                       P_G2L         = float(0.00000),
                       P_B2L         = float(0.00000),
                       buy_price     = float(0.00000),
                       sell_price = float(0.0000))
    n_ut = 1
    n_lt = 1
    
    #first row
    UT = rule_based.iloc[0 : 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation='higher')
    LT = 0
    
    if rule_based.at[0,"deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[0,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < SOC_MAX):
                   rule_based.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[0,"SOC_start"],CHARGE_RATE])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[0,"SOC_start"] > SOC_MIN:
    
                    #if rule_based.at[i,"deficit"] < DISCHARGE_RATE:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"]- SOC_MIN])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,DISCHARGE_RATE, rule_based.at[i,"SOC_start"]- SOC_MIN])
                    rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"deficit"], rule_based.at[0,"SOC_start"]- SOC_MIN, DISCHARGE_RATE])
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"] - rule_based.at[0,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"]
                    
        # surplus side            
    else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[0,"SOC_start"] < SOC_MAX:
    
                # Is surplus < Charge rate
                if rule_based.at[0,"surplus"] < CHARGE_RATE:
                    
                    rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"surplus"], SOC_MAX - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"] - rule_based.at[0,"P_P2B"]
                    
                    if rule_based.at[0,"price_eur_per_mwh"] < LT:
                        rule_based.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[0,"P_P2B"], SOC_MAX - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[0,"P_P2B"] = __builtins__.min(CHARGE_RATE, SOC_MAX - rule_based.at[0,"SOC_start"])
                    rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"] - rule_based.at[0,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"]
    
    #P_P2L
    
    #P_P2G
    #P_G2B
    #P_G2L
    
    
    rule_based.at[0,"SOC_end"] = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
    rule_based.at[0,"buy_price"] = (rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"])*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    rule_based.at[0,"sell_price"] = rule_based.at[0,"P_P2G"]*rule_based.at[0,"price_eur_per_mwh"]*0.00025
    
    for i in range(1, rule_based.shape[0]):
        # setting SOC 
        rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
        # calculating thresholds
        if (n_ut % 96 == 0) & (n_ut != rule_based.shape[0]):
            UT = rule_based.iloc[i : i + 96]['price_eur_per_mwh'].quantile(percentiles[get_season(rule_based.at[i,"timestamp"])], interpolation='higher')
        if (n_lt % 96) & (n_lt >= 1440):
            LT = rule_based.iloc[i - 1439 : i]['price_eur_per_mwh'].quantile(0.05, interpolation='lower')
        n_ut += 1
        n_lt += 1
    
        # comparing demand and production
        # deficit side
        if rule_based.at[i,"deficit"] > 0:
            #P_P2B = 0
            #P_P2G = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[i,"price_eur_per_mwh"] < UT:
                #P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < SOC_MAX):
                   rule_based.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[i,"SOC_start"],CHARGE_RATE])
                    #else P_G2B = 0
            # expensive
            else:
                # P_G2B = 0
    
                # BESS not empty
                if rule_based.at[i,"SOC_start"] > SOC_MIN:
    
                    #if rule_based.at[i,"deficit"] < DISCHARGE_RATE:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"]- SOC_MIN])
                    #
                    #else:
                    #    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,DISCHARGE_RATE, rule_based.at[i,"SOC_start"]- SOC_MIN])
                    rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"]- SOC_MIN, DISCHARGE_RATE])
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"] - rule_based.at[i,"P_B2L"]
                #BESS is empty
                else:
                    #P_B2L = 0
                    rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"]
                    
        # surplus side            
        else:
            #P_G2L = 0
            #P_B2L = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
    
            # is BESS full?
            if rule_based.at[i,"SOC_start"] < SOC_MAX:
    
                # Is surplus < Charge rate
                if rule_based.at[i,"surplus"] < CHARGE_RATE:
                    
                    rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"surplus"], SOC_MAX - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"] - rule_based.at[i,"P_P2B"]
                    
                    if rule_based.at[i,"price_eur_per_mwh"] < LT:
                        rule_based.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[i,"P_P2B"], SOC_MAX - rule_based.at[i,"SOC_start"] - rule_based.at[i,"P_P2B"])
                        # else P_G2B = 0
                
                # surplus bigger
                else:
                    # P_G2B = 0
                    rule_based.at[i,"P_P2B"] = __builtins__.min(CHARGE_RATE, SOC_MAX - rule_based.at[i,"SOC_start"])
                    rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"] - rule_based.at[i,"P_P2B"]
                    
            # BESS is full
            else:
                #P_P2B = 0
                #P_G2B = 0
                rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"]
    
        #executing the calculations
        rule_based.at[i,"SOC_end"]       = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
        rule_based.at[i,"buy_price"]     = (rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"])*rule_based.at[i,"price_eur_per_mwh"]*0.00025
        rule_based.at[i,"sell_price"] = rule_based.at[i,"P_P2G"]*rule_based.at[i,"price_eur_per_mwh"]*0.00025
                
    UT=0
    LT=0
    r_total_generation = rule_based["production_kwh"].sum()
    r_total_export = rule_based["P_P2G"].sum()
    r_self_consumption_rate = 1 - r_total_export/r_total_generation
    
    r_total_consumption = rule_based["consumption_kwh"].sum()
    r_total_import = rule_based["P_G2L"].sum() + rule_based["P_G2B"].sum()
    r_self_sufficiency_rate = 1 - r_total_import/r_total_consumption
    
    r_total_cost = rule_based["buy_price"].sum()
    r_total_revenue = rule_based["sell_price"].sum()
    r_net_revenue = r_total_revenue - r_total_cost
    
    string_percentiles = ','.join([str(value) for value in percentiles.values()])
    dataset.append((string_percentiles, float(r_self_consumption_rate), float(r_self_sufficiency_rate), float(r_total_cost), float(r_total_revenue), float(r_net_revenue)))

    c = c +1
    print(c)
    #print(percentiles.values())
    #print(f"Self consumption rate: {r_self_consumption_rate}")
    #print(f"Self sufficiency rate: {r_self_sufficiency_rate}")
    #print(f"Total cost: {r_total_cost}")
    #print(f"Total revenue: {r_total_revenue}")
    #print(f"Net revenue: {r_net_revenue}")

df = spark.createDataFrame(dataset, columns)
df.orderBy("total_cost").show()

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
+---------------+---------------------+---------------------+------------------+-------------------+-------------------+
|    percentiles|self_consumption_rate|self_sufficiency_rate|        total_cost|      total_revenue|        net_revenue|
+---------------+---------------------+---------------------+------------------+-------------------+-------------------+
|0.5,0.1,0.1,0.5|   0.9510962344716383|  0.16252004733649184|20.495742785741726|0.06514390439379714| -20.43059888134793|
|0.5,0.5,0.1,0.5|   0.9500935869583007|  0.16234871838425402|20.496148850873805|0.06735497367630072|-20.428793877197503|
|0.1,0.1,0.1,0.5|   0.9510962344716383|  0.16252004733649184|20.499352292158715|0.06514390439379714| -20.43420838776492|
|0.1,0.5,0.1,0.5|   0.9500935869583007|  0.16234871838425402|20.499758357290798|0.06735497367630072|-20.432403383614496|
|0.5,0.1,0.5,0.5|    0.950858562534215|  0.16247943477477567|20.770666234640267|0.06550475081227751| -20.705161483

In [ ]:
df = df.repartition(1)
df.write.format("csv").option("header","true").option("delimiter",";").mode("overwrite").save("../output/percentiles_177k_10000_kwh_per_y.csv")

In [ ]:
df.orderBy("total_cost").show()

# Linear Programming 

In [20]:
import pyomo.environ as pyo
import pandas
import numpy as np
from pyomo.opt import (SolverFactory, TerminationCondition)

**algorithm 1**

In [224]:
#dates = process
#dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
#dates["date"] = dates["timestamp"].dt.date
#days =  dates['date'].unique().tolist()

#data = process
#data["timestamp"] = pandas.to_datetime(data["timestamp"])
#data = data[data["timestamp"].dt.date == days[365]]
#data.reset_index(drop=True, inplace=True)
#t = data.shape[0]

def cost_model(data, t, start):
    con= {}
    gen= {}
    price = {}
    for i in range(0,t):
        con[i+1] = data["consumption_kwh"][i]
        gen[i+1] = data["production_kwh"][i]
        price[i+1] = data["price_eur_per_mwh"][i]*0.00025
    
    model = pyo.ConcreteModel("(E)")

    model.T = pyo.RangeSet(t)
    
    model.cin = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.cout = pyo.Var(model.T, within= pyo.NonNegativeReals)
    model.ch = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(0,CHARGE_RATE))
    model.dch = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(0,DISCHARGE_RATE)) 
    model.soc = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(SOC_MIN, SOC_MAX))
    model.soc0 = pyo.Param(within= pyo.NonNegativeReals, default=0, initialize=start)

    model.y_ch = pyo.Var(model.T, within=pyo.Binary)
    model.y_dch = pyo.Var(model.T, within=pyo.Binary)

    
    model.con = pyo.Param(model.T, initialize=con)
    model.gen = pyo.Param(model.T, initialize=gen)
    model.price = pyo.Param(model.T, initialize=price)

    def obj_rule(m):
        #return __builtins__.sum(m.cin[t]*m.price[t] - m.cout[t]*m.price[t] for t in m.T)
        return __builtins__.sum(m.cin[t]*m.price[t] for t in m.T)
        #return __builtins__.sum(m.cin[t]*m.price[t] for t in m.T)*0.8 + __builtins__.sum(m.cin[t]*m.price[t] - m.cout[t]*m.price[t] for t in m.T)*0.2
    model.obj = pyo.Objective(rule=obj_rule)

    def battery_link_rule(m,t):
        if t == m.T.first():
            return m.soc[t] == m.soc0 -m.dch[t] + m.ch[t]
        return m.soc[t] == m.soc[t-1] + m.ch[t] - m.dch[t]
    model.battery_link_con = pyo.Constraint(model.T, rule= battery_link_rule)
    
    def charge_rate_rule(m,t):
        return m.ch[t] <= CHARGE_RATE*m.y_ch[t]
    model.charge_rate_con = pyo.Constraint(model.T, rule= charge_rate_rule)
    def discharge_rate_rule(m,t):
        return m.dch[t] <= DISCHARGE_RATE*m.y_dch[t]
    model.discharge_rate_con = pyo.Constraint(model.T, rule= discharge_rate_rule)
    def ch_dch_rule(m, t):
        return m.y_ch[t] + m.y_dch[t] <= 1
    model.ch_dch_con = pyo.Constraint(model.T, rule= ch_dch_rule)


    def balance_rule(m,t):
        return m.ch[t] + m.con[t] + m.cout[t] == m.dch[t] + m.gen[t] + m.cin[t]
    model.balance_con = pyo.Constraint(model.T, rule= balance_rule)

    return model


### SELECT a solver:

#solver = pyo.SolverFactory('glpk')
#solver = pyo.SolverFactory('cbc')

#result = solver.solve(model, tee=True)

#pyo.assert_optimal_termination(result)
#print(pyo.value(model.obj))

#model.con.pprint()
#model.gen.pprint()
#model.price.pprint()
#model.ch.pprint()
#model.dch.pprint()
#model.soc.pprint()
#model.cin.pprint()
#model.cout.pprint()

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpifweoysx.glpk.raw --wglp /tmp/tmps_7lb8vn.glpk.glp --cpxlp
 /tmp/tmpe517plhl.pyomo.lp
Reading problem data from '/tmp/tmpe517plhl.pyomo.lp'...
/tmp/tmpe517plhl.pyomo.lp:1296485: warning: lower bound of variable 'x175199' redefined
/tmp/tmpe517plhl.pyomo.lp:1296485: warning: upper bound of variable 'x175199' redefined
175200 rows, 245280 columns, 490559 non-zeros
70080 integer variables, all of which are binary
1366565 lines were read
Writing problem data to '/tmp/tmps_7lb8vn.glpk.glp'...
1261438 lines were written
GLPK Integer Optimizer 5.0
175200 rows, 245280 columns, 490559 non-zeros
70080 integer variables, all of which are binary
Preprocessing...
175200 rows, 210239 columns, 455518 non-zeros
70080 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.250e+01  ratio =  1.250e+01
GM: min|aij| =  9.150e-01  max|aij| =  1.093e+00  ratio =  1.194e+00
EQ: m

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f4a3655ff10>>
Traceback (most recent call last):
  File "/home/student/.local/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 

KeyboardInterrupt



**algorithm 2**

In [198]:
def cost_model_2(data, t, start):
    con= {}
    gen= {}
    price = {}
    for i in range(0,t):
        con[i+1] = data["consumption_kwh"][i]
        gen[i+1] = data["production_kwh"][i]
        price[i+1] = data["price_eur_per_mwh"][i]*0.00025
    
    model = pyo.ConcreteModel("(E)")

    model.T = pyo.RangeSet(t)
    
    model.cin = pyo.Var(model.T, within=pyo.NonNegativeReals)
    model.cout = pyo.Var(model.T, within= pyo.NonNegativeReals)
    model.ch = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(0,CHARGE_RATE))
    model.dch = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(0,DISCHARGE_RATE)) 
    model.soc = pyo.Var(model.T, within= pyo.NonNegativeReals, bounds=(SOC_MIN, SOC_MAX))
    model.soc0 = pyo.Param(within= pyo.NonNegativeReals, default=0, initialize=start)
    
    model.con = pyo.Param(model.T, initialize=con)
    model.gen = pyo.Param(model.T, initialize=gen)
    model.price = pyo.Param(model.T, initialize=price)

    def obj_rule(m):
        #return __builtins__.sum(m.cin[t]*m.price[t] - m.cout[t]*m.price[t] for t in m.T)
        return __builtins__.sum(m.cin[t]*m.price[t] for t in m.T)
        #return __builtins__.sum(m.cin[t]*m.price[t] for t in m.T)*0.8 + __builtins__.sum(m.cin[t]*m.price[t] - m.cout[t]*m.price[t] for t in m.T)*0.2
    model.obj = pyo.Objective(rule=obj_rule)

    def battery_link_rule(m,t):
        if t == m.T.first():
            return m.soc[t] == m.soc0 -m.dch[t] + m.ch[t]
        return m.soc[t] == m.soc[t-1] + m.ch[t] - m.dch[t]
    model.battery_link_con = pyo.Constraint(model.T, rule= battery_link_rule)
    
    def balance_rule(m,t):
        return m.ch[t] + m.con[t] + m.cout[t] == m.dch[t] + m.gen[t] + m.cin[t]
    model.balance_con = pyo.Constraint(model.T, rule= balance_rule)

    return model

#model = cost_model(data, t, START)

### SELECT a solver:

#solver = pyo.SolverFactory('glpk')
#solver = pyo.SolverFactory('cbc')

#result = solver.solve(model, tee=True)

#pyo.assert_optimal_termination(result)
#print(pyo.value(model.obj))

#model.con.pprint()
#model.gen.pprint()
#model.price.pprint()
#model.ch.pprint()
#model.dch.pprint()
#model.soc.pprint()
#model.cin.pprint()
#model.cout.pprint()

# Backup algorithm


**Greedy**

In [72]:
#dates = process
#dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
#dates["date"] = dates["timestamp"].dt.date
#days =  dates['date'].unique().tolist()

#data = process
#data["timestamp"] = pandas.to_datetime(data["timestamp"])
#data = data[data["timestamp"].dt.date == days[365]]
#data.reset_index(drop=True, inplace=True)

def backup_greedy(data, start):

# calculating energy surplus and deficit from the difference of production and consumption,
# SOC colums are added for the beggining and end of 15 minute intervals, setting default value to 0. 
    data = data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                       deficit   = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                       SOC_start = float(0.00000),
                       SOC_end   = float(0.00000))


    data.at[0,"SOC_start"] = start

#calculating the amount we can store of the surplus generated
    if data.at[0,"surplus"] > 0:
       data.at[0,"SOC_end"] = __builtins__.min(data.at[0,"SOC_start"] + data.at[0,"surplus"]\
                                                   ,data.at[0,"SOC_start"] + CHARGE_RATE\
                                                   ,SOC_MAX)
            
#calculating how much of the deficit we can cover from our battery
    else:
        data.at[0,"SOC_end"] = __builtins__.max(data.at[0,"SOC_start"] - data.at[0,"deficit"]\
                                                   ,data.at[0,"SOC_start"] - DISCHARGE_RATE\
                                                   ,SOC_MIN)

    for i in range(1,data.shape[0]):
        data.at[i,"SOC_start"] = data.at[i-1,"SOC_end"]
        if data.at[i,"surplus"] > 0:
            data.at[i,"SOC_end"] = __builtins__.min(data.at[i,"SOC_start"] + data.at[i,"surplus"]\
                                                   ,data.at[i,"SOC_start"] + CHARGE_RATE\
                                                   ,SOC_MAX)
        else:
            data.at[i,"SOC_end"] = __builtins__.max(data.at[i,"SOC_start"] - data.at[i,"deficit"]\
                                                   ,data.at[i,"SOC_start"] - DISCHARGE_RATE\
                                                   ,SOC_MIN)
    
    # from the data previously calculated, we determine how much energy we used from our own storage and from the grid,
    # furthermore how much energy we feed our system or the grid.
    # last we calculate the energy price and the revenue. 
    data = data.assign(dch   = lambda x: np.where(x["SOC_start"] - x["SOC_end"] > 0, x["SOC_start"] - x["SOC_end"], 0),
                       ch    = lambda x: np.where(x["SOC_end"] - x["SOC_start"] > 0, x["SOC_end"] - x["SOC_start"], 0),
                       c_in = lambda x: x["deficit"]-x["dch"],
                       c_out  = lambda x: x["surplus"]-x["ch"],
                       price = lambda x: (x["c_out"]*x["price_eur_per_mwh"]-x["c_in"]*x["price_eur_per_mwh"])*0.00025) 
    output = {
                "cin":data["c_in"].tolist(),
                "cout":data["c_out"].tolist(),
                "soc":data["SOC_end"].tolist(),
                "ch":data["ch"].tolist(),
                "dch":data["dch"].tolist()
             }
    return output

#output = backup_greedy(data, START)

**Rule Based**

In [86]:
percentiles = {"Winter":0.1,"Spring":0.1,"Summer":0.1,"Autumn":0.1}
def backup_rule(data, start):
    rule_based = data.assign(surplus = lambda x: np.where(x["production_kwh"] - x["consumption_kwh"] > 0, (x["production_kwh"] - x["consumption_kwh"]), 0),
                            deficit  = lambda x: np.where(x["consumption_kwh"] - x["production_kwh"] > 0, (x["consumption_kwh"] - x["production_kwh"]), 0),
                            SOC_start     = float(0.00000),
                            SOC_end       = float(0.00000),
                            P_P2L         = float(0.00000),
                            P_P2B         = float(0.00000),
                            P_P2G         = float(0.00000),
                            P_G2B         = float(0.00000),
                            P_G2L         = float(0.00000),
                            P_B2L         = float(0.00000),
                            buy_price     = float(0.00000),
                            sell_price = float(0.0000))
    
    #first row
    rule_based["SOC_start"] = start
    UT = rule_based["price_eur_per_mwh"].quantile(percentiles[get_season(rule_based.at[0,"timestamp"])], interpolation="higher")
    LT = rule_based["price_eur_per_mwh"].quantile(0.05, interpolation="lower")
    
    # comparing demand and prodcuction
    # deficit side
    if rule_based.at[0,"deficit"] > 0:
        
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[0,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[0,"price_eur_per_mwh"] < LT) & (rule_based.at[0,"SOC_start"] < SOC_MAX):
                   rule_based.at[0,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[0,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based.at[0,"P_B2L"] = __builtins__.min([rule_based.at[0,"deficit"], rule_based.at[0,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based.at[0,"P_G2L"] = rule_based.at[0,"deficit"] - rule_based.at[0,"P_B2L"]
                    
                    
    # surplus side            
    else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based.at[0,"P_P2L"] = rule_based.at[0,"consumption_kwh"]
            rule_based.at[0,"P_P2B"] = __builtins__.min(rule_based.at[0,"surplus"]\
                                                        ,SOC_MAX - rule_based.at[0,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based.at[0,"P_P2G"] = rule_based.at[0,"surplus"] - rule_based.at[0,"P_P2B"]
        
            if rule_based.at[0,"price_eur_per_mwh"] < LT:
                        rule_based.at[0,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[0,"P_P2B"], SOC_MAX - rule_based.at[0,"SOC_start"] - rule_based.at[0,"P_P2B"])
            ##else P_G2B = 0
    
    # calculating prices and SOC
    rule_based.at[0,"SOC_end"]    = rule_based.at[0,"SOC_start"] + rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"] - rule_based.at[0,"P_B2L"]
    rule_based.at[0,"c_in"]  = rule_based.at[0,"P_G2B"] + rule_based.at[0,"P_G2L"]
    rule_based.at[0,"c_out"] = rule_based.at[0,"P_P2G"]
    rule_based.at[0,"ch"] = rule_based.at[0,"P_P2B"] + rule_based.at[0,"P_G2B"]
    rule_based.at[0,"dch"] = rule_based.at[0,"P_B2L"]
    
    # repeat it for each row
    for i in range(1, rule_based.shape[0]):
        
        # setting SOC 
        rule_based.at[i,"SOC_start"] = rule_based.at[i-1,"SOC_end"]
        
        # comparing demand and production
        # deficit side
        if rule_based.at[i,"deficit"] > 0:
            ##P_P2B = 0
            ##P_P2G = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"production_kwh"]
            
            # cheaper than UT
            if rule_based.at[i,"price_eur_per_mwh"] < UT:
                ##P_B2L = 0
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"]
    
                #cheaper than LT and there is space
                if (rule_based.at[i,"price_eur_per_mwh"] < LT) & (rule_based.at[i,"SOC_start"] < SOC_MAX):
                   rule_based.at[i,"P_G2B"] = __builtins__.min([SOC_MAX - rule_based.at[i,"SOC_start"],CHARGE_RATE])
                ##else P_G2B = 0
        
            # expensive
            else:
                ##P_G2B = 0
    
                rule_based.at[i,"P_B2L"] = __builtins__.min([rule_based.at[i,"deficit"], rule_based.at[i,"SOC_start"] - SOC_MIN, DISCHARGE_RATE])
                rule_based.at[i,"P_G2L"] = rule_based.at[i,"deficit"] - rule_based.at[i,"P_B2L"]
                
                    
        # surplus side            
        else:
            ##P_G2L = 0
            ##P_B2L = 0
            rule_based.at[i,"P_P2L"] = rule_based.at[i,"consumption_kwh"]
            rule_based.at[i,"P_P2B"] = __builtins__.min(rule_based.at[i,"surplus"]\
                                                        ,SOC_MAX - rule_based.at[i,"SOC_start"]\
                                                        ,CHARGE_RATE)
            rule_based.at[i,"P_P2G"] = rule_based.at[i,"surplus"] - rule_based.at[i,"P_P2B"]
        
            if rule_based.at[i,"price_eur_per_mwh"] < LT:
                        rule_based.at[i,"P_G2B"] = __builtins__.min(CHARGE_RATE - rule_based.at[i,"P_P2B"], SOC_MAX - rule_based.at[i,"SOC_start"] - rule_based.at[0,"P_P2B"])
            ##else P_G2B = 0
            
        #executing the calculations
        rule_based.at[i,"SOC_end"]    = rule_based.at[i,"SOC_start"] + rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"] - rule_based.at[i,"P_B2L"]
        rule_based.at[i,"c_in"]  = rule_based.at[i,"P_G2B"] + rule_based.at[i,"P_G2L"]
        rule_based.at[i,"c_out"] = rule_based.at[i,"P_P2G"]
        rule_based.at[i,"ch"] = rule_based.at[i,"P_P2B"] + rule_based.at[i,"P_G2B"]
        rule_based.at[i,"dch"] = rule_based.at[i,"P_B2L"]

    output = {
                "cin":rule_based["c_in"].tolist(),
                "cout":rule_based["c_out"].tolist(),
                "soc":rule_based["SOC_end"].tolist(),
                "ch":rule_based["ch"].tolist(),
                "dch":rule_based["dch"].tolist()
             }
        
    return output
    

# Running the LP algorithm

In [217]:
start = START

error_count=0
# collecting all dates into a list of unique days.
dates = process
dates["timestamp"] = pandas.to_datetime(dates["timestamp"])
dates["date"] = dates["timestamp"].dt.date
days =  dates['date'].unique().tolist()


linear_programming = process
linear_programming["timestamp"] = pandas.to_datetime(linear_programming["timestamp"])

SOC = []
ch = []
dch = []
c_in = []
c_out = []
#iterate over each day from the database
for day in days:
    
    
    data = linear_programming[linear_programming["timestamp"].dt.date == day]
    data.reset_index(drop=True, inplace=True)
    t = data.shape[0]
    
    model = cost_model(data, t, start)
    #model = cost_model_2(data, t, start)
    solver = pyo.SolverFactory('glpk')
    result = solver.solve(model)
    if pyo.check_optimal_termination(result):
        SOC.extend([model.soc[t].value for t in model.T])
        ch.extend([model.ch[t].value for t in model.T])
        dch.extend([model.dch[t].value for t in model.T])
        c_in.extend([model.cin[t].value for t in model.T])
        c_out.extend([model.cout[t].value for t in model.T])
    else:
        output = backup_greedy(data, start)
        #output = backup_rule(data, start)
        
        SOC.extend(output["soc"])
        ch.extend(output["ch"])
        dch.extend(output["dch"])
        c_in.extend(output["cin"])
        c_out.extend(output["cout"])
        
        error_count += 1
    start = SOC[-1]
print("Job's done!")
print(f"errors: {error_count}")

Job's done!
errors: 0


In [218]:
linear_programming["soc"] = SOC
linear_programming["ch"] = ch
linear_programming["dch"] = dch
linear_programming["c_in"] = c_in
linear_programming["c_out"] = c_out
linear_programming["buy_price"] = linear_programming["c_in"]*linear_programming["price_eur_per_mwh"]*0.00025
linear_programming["sell_price"] = linear_programming["c_out"]*linear_programming["price_eur_per_mwh"]*0.00025

In [219]:
lp_total_generation = linear_programming["production_kwh"].sum()
lp_total_export = linear_programming["c_out"].sum()
lp_self_consumption_rate = 1 - lp_total_export/lp_total_generation

lp_total_consumption = linear_programming["consumption_kwh"].sum()
lp_total_import = linear_programming["c_in"].sum()
lp_self_sufficiency_rate = 1 - lp_total_import/lp_total_consumption

lp_total_cost = linear_programming["buy_price"].sum()
lp_total_revenue = linear_programming["sell_price"].sum()
lp_net_revenue = lp_total_revenue - lp_total_cost

print(f"Total generation: {lp_total_generation} kwh")
print(f"Total consumption: {lp_total_consumption} kwh")
print(f"Self consumption rate: {lp_self_consumption_rate}")
print(f"Self sufficiency rate: {lp_self_sufficiency_rate}")
print(f"Total cost: {lp_total_cost} €")
print(f"Total revenue: {lp_total_revenue} €")
print(f"Net revenue: {lp_net_revenue} €")


Total generation: 38.948577880859375 kwh
Total consumption: 172.95486450195312 kwh
Self consumption rate: 0.8374472739456609
Self sufficiency rate: 0.18858895595520286
Total cost: 2.0279349840421133 €
Total revenue: 0.11693221036328837 €
Net revenue: -1.9110027736788249 €


# Testing area

In [73]:
header = ["production_kwh", "consumption_kwh","price_eur_per_mwh"]
data = [
        (100,110,10),
        (90,120,12),
        (95,90,9),
        (105,65,5),
        (50,80,12),
        (100,100,20)
        ]
tester = pandas.DataFrame(data, columns=header)
start = 0
results = backup_greedy(tester, start)
tester["cin"] = results["cin"]
tester["cout"] = results["cout"]
tester["soc"] = results["soc"]
tester["ch"] = results["ch"]
tester["dch"] = results["dch"]
print(tester)

   production_kwh  consumption_kwh  price_eur_per_mwh   cin  cout   soc    ch  \
0             100              110                 10  10.0   0.0   0.0   0.0   
1              90              120                 12  30.0   0.0   0.0   0.0   
2              95               90                  9   0.0   0.0   5.0   5.0   
3             105               65                  5   0.0  27.5  17.5  12.5   
4              50               80                 12  17.5   0.0   5.0   0.0   
5             100              100                 20   0.0   0.0   5.0   0.0   

    dch  
0   0.0  
1   0.0  
2   0.0  
3   0.0  
4  12.5  
5   0.0  


In [108]:
df = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../output/percentiles_177k_10000_kwh_per_y/177k.csv")

In [116]:
df.orderBy(desc(col("total_cost"))).show()

+---------------+---------------------+---------------------+------------------+------------------+------------------+
|    percentiles|self_consumption_rate|self_sufficiency_rate|        total_cost|     total_revenue|       net_revenue|
+---------------+---------------------+---------------------+------------------+------------------+------------------+
|0.5,0.5,0.5,0.5|   0.7736904531405625|   0.5734676423505014|11.124873912805848| 2.470850818898003|-8.654023093907846|
|0.4,0.5,0.5,0.5|   0.7736904531405625|   0.5734676423505014|11.059080816108834| 2.470850818898003|-8.588229997210831|
|0.3,0.5,0.5,0.5|   0.7736904531405625|   0.5734676423505014|11.056868437938066| 2.470850818898003|-8.586017619040064|
|0.1,0.5,0.5,0.5|   0.7736904531405625|   0.5734676423505014|11.053931736162076| 2.470850818898003|-8.583080917264073|
|0.2,0.5,0.5,0.5|   0.7736904531405625|   0.5734676423505014|11.053131927546337| 2.470850818898003|-8.582281108648335|
|0.5,0.5,0.4,0.5|   0.7747772655095619|     0.57